# ReefScan-ID: NOAA bleaching dataset check

Check of the NOAA PIFSC ESD Coral Bleaching dataset before it is used for stage 2. Run in Google Colab; no GPU is needed.

### Setup: libraries

In [ ]:
import huggingface_hub
import pandas as pd

print("huggingface_hub version: {}".format(huggingface_hub.__version__))
print("pandas version: {}".format(pd.__version__))

### Setup: Google Drive

In [ ]:
import os
from google.colab import drive

drive.mount("/content/drive")

base_dir = "/content/drive/MyDrive/ReefScan-ID"
noaa_dir = base_dir + "/RawData/NOAA-PIFSC-ESD-CORAL-Bleaching-Dataset"
print("NOAA folder found: {}".format(os.path.isdir(noaa_dir)))
print("Contents: {}".format(sorted(os.listdir(noaa_dir))))

## Section 1: Do the labels load?

Question: does every image in the Drive copy of the dataset get one of the two labels (CORAL and CORAL_BL) from its folder name, with the counts the folders on the Hub hold?

Checks: count the image files in each split and class folder of the repository on the Hub, list the images in the Drive folder with the split and label taken from the path, then compare the counts per folder and print the label counts per split.

### Files in the repository

In [ ]:
from collections import Counter
from huggingface_hub import list_repo_files

repo_id = "NMFS-OSI/NOAA-PIFSC-ESD-CORAL-Bleaching-Dataset"
image_endings = (".jpg", ".jpeg", ".png")

files = list_repo_files(repo_id, repo_type="dataset")
image_files = [name for name in files if name.lower().endswith(image_endings)]
other_files = [name for name in files if not name.lower().endswith(image_endings)]

print("Files in the repository: {}".format(len(files)))
print("Image files: {}".format(len(image_files)))
print("Other files: {}".format(other_files[:20]))

folder_counts = Counter()
for name in image_files:
    folder_counts["/".join(name.split("/")[:-1])] += 1

for folder in sorted(folder_counts):
    print("{}: {} images".format(folder, folder_counts[folder]))

### Images in the Drive folder

In [ ]:
import time

start = time.time()
rows = []
skipped = []
for root, dirs, names in os.walk(noaa_dir):
    if ".git" in dirs:
        dirs.remove(".git")
    for name in names:
        relative = os.path.relpath(os.path.join(root, name), noaa_dir).replace(os.sep, "/")
        parts = relative.split("/")
        if name.lower().endswith(image_endings) and len(parts) == 3:
            rows.append({"split": parts[0], "label": parts[1], "path": relative})
        else:
            skipped.append(relative)

noaa = pd.DataFrame(rows)
print("Listing took {:.1f} seconds".format(time.time() - start))
print("Images found: {}".format(len(noaa)))
print("Files skipped: {}, first ones: {}".format(len(skipped), skipped[:20]))
print("Splits: {}".format(sorted(noaa["split"].unique())))
print("Labels: {}".format(sorted(noaa["label"].unique())))
print(noaa.head(5))

### Row counts against the folders

In [ ]:
drive_counts = Counter(noaa["split"] + "/" + noaa["label"])

for folder in sorted(set(folder_counts) | set(drive_counts)):
    print("{}: {} on the Hub, {} on Drive, same: {}".format(
        folder, folder_counts[folder], drive_counts[folder], folder_counts[folder] == drive_counts[folder]))
print("Total: {} on the Hub, {} on Drive".format(sum(folder_counts.values()), len(noaa)))

print(pd.crosstab(noaa["split"], noaa["label"]))

**Observation**: